In [1]:
import os
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import json
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
import numpy as np

In [2]:
USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project), Mozilla/5.0 (Windows NT 10.0; Win64; x64), Chrome/91.0.4472.124 Safari/537.36")

In [3]:
urls_for_load = Path('../data/web/')

In [4]:
def load_urls(path: Path):
    urls = []
    for file in path.glob("SpeewaldRegion.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry)

    return urls
urls = load_urls(urls_for_load)
print(f"{len(urls)} Seiten geladen")

13 Seiten geladen


In [5]:
headers = {"User-Agent": USER_AGENT}

data = []

for entry in urls:
    url = entry["url"]
    category = entry["content_type"]

    html = requests.get(url, headers=headers).text
    soup = BeautifulSoup(html, "html.parser")

    for div in soup.select(".suchergebnisseListeBox"):
        lines = div.get_text("\n", strip=True).split("\n") #setzt die Zeilenumbrüche zwischen HTML-Elemente. Split mach eine Liste von Zeilen
        data.append({
        "name": lines[0],
        "ort": lines[1] if len(lines) > 1 else "",
        "category":  category,
        "url": url
    })
        print(f"Name: {lines[0]}{'name'}")
        print(f"Ort: {lines[1]}{'ort'}")
        print(f"Category: {category}\n")


Name: Kahnfahrt Wolfgang Gahl mit Kahnfahrschulename
Ort: An der Dolzke 7b in 03222 Lübbenau (Spreewald)ort
Category: kahnfahrt

Name: Kahnfahrt am Hafen zur alten Aalreusename
Ort: Waldschlösschenstraße 17 in 03096 Burg (Spreewald)ort
Category: kahnfahrt

Name: Kahnfahrt mit Guido Sieg & Fährfrau Antjename
Ort: Thomas-Müntzer-Straße 1 in 15907 Lübben (Spreewald)ort
Category: kahnfahrt

Name: Kahngenuss Spreewaldkahnfahrtenname
Ort: Kirchstr. 4 in 15913 Straupitz (Spreewald)ort
Category: kahnfahrt

Name: Kahnfahrt Siegmund Lehmannname
Ort: in 03096 Burg (Spreewald)ort
Category: kahnfahrt

Name: Kahnfahrt am Hotel Am Spreebogenname
Ort: Ringchaussee 140 in 03096 Burg (Spreewald)ort
Category: kahnfahrt

Name: Manuelas Kahnfahrt in Lehdename
Ort: An der Quodda 2 in 03222 Lübbenau /Spreewaldort
Category: kahnfahrt

Name: Kahnfahrt am Bootshaus Leinewebername
Ort: Hauptstraße 1 in 03096 Burg (Spreewald)ort
Category: kahnfahrt

Name: Kahnfährmann Karl-Heinz Wendlandname
Ort: Dammstraße 72 in

In [6]:
#embeddings erzeugen
from sentence_transformers import SentenceTransformer
model = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(model)
for entry in data:
    text = f"{entry['name']} in {entry['ort']} ({entry['category']})"
    entry["text"] = text
    entry["embedding"] = embedding_model.encode(text).tolist()


In [7]:
connections.connect("default", host="localhost", port="19530")

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="url", dtype=DataType.VARCHAR, max_length=1024)
]

schema = CollectionSchema(fields, description="Spreewald Data")

# Collection erstellen
collection_name = "spreewald_collection"
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

collection = Collection(name=collection_name, schema=schema)

# Einfügen
collection.insert([
    [e["embedding"] for e in data],
    [e["text"] for e in data],
    [e["url"] for e in data]
])

(insert count: 119, delete count: 0, upsert count: 0, timestamp: 464917374315003914, success count: 119, err count: 0

In [8]:
#index erstellen
index_params = {
    "index_type": "HNSW",
    "metric_type": "IP",
    "params": {"M": 16, "efConstruction": 200}
}
collection.create_index(field_name="embedding", index_params=index_params)
print(f"Index für Collection '{collection_name}' erstellt.")

collection.load()

Index für Collection 'spreewald_collection' erstellt.


In [9]:
#Semantische Suche in Milvus
def search(query: str, k: int = 5):
    query_embedding = embedding_model.encode(query)
    query_embedding = query_embedding.astype(np.float32).tolist()  # Milvus erwartet FLOAT32-Format

    results = collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param={"metric_type": "IP", "params": {"ef": 50}},
        limit=k,
        output_fields=["text", "url"]
    )

    output = []

    for hit in results[0]:
            output.append({
                "score": float(hit.score),
                "text": hit.entity.get("text"),
                "url": hit.entity.get("url")
            })
    return output

In [10]:
#Test
query = "Kahnfahrt in Lübbenau"
results = search(query, k=5)
for i, res in enumerate(results):
    print(f"\n--- Treffer {i+1} ---")
    print(f"Score: {res['score']:.4f}")
    print(f"Text: {res['text']}")
    print(f"URL: {res['url']}")


--- Treffer 1 ---
Score: 10.4160
Text: Kahnfahrt Bernds Kahnfahrten in Lehnigksberger Weg 6 in 15907 Lübben (kahnfahrt)
URL: https://www.spreewald-info.de/kahnfahrt/spreewald-kahnfahrt-luebben/

--- Treffer 2 ---
Score: 10.4160
Text: Kahnfahrt Bernds Kahnfahrten in Lehnigksberger Weg 6 in 15907 Lübben (kahnfahrt)
URL: https://www.spreewald-info.de/kahnfahrt/haefen/

--- Treffer 3 ---
Score: 9.6660
Text: Jörg´s Kahnfahrten in Lehnigksberger Weg 1 in 15907 Lübben (Spreewald) (kahnfahrt)
URL: https://www.spreewald-info.de/kahnfahrt/spreewald-kahnfahrt-luebben/

--- Treffer 4 ---
Score: 9.6660
Text: Jörg´s Kahnfahrten in Lehnigksberger Weg 1 in 15907 Lübben (Spreewald) (kahnfahrt)
URL: https://www.spreewald-info.de/kahnfahrt/haefen/

--- Treffer 5 ---
Score: 9.4111
Text: Museum Schloss Lübben in Ernst-von-Houwald-Damm 14 in 15907 Lübben (Spreewald) (museum)
URL: https://www.spreewald-info.de/ausflugsziele/museen/
